# Train LSTM PPO - Trial 5: Improved Early Stopping

**Trial:** #5 (LSTM=50, freq=5000, patience=3)

**Strategy:** Test if less frequent evaluation + more aggressive stopping reduces overfitting

**Key Changes from Previous Trials:**
- LSTM_HIDDEN_SIZE: 50 (Trial 2 best config, not 75)
- EVAL_FREQ: 5000 (was 2500 - DOUBLED, less frequent checking)
- EARLY_STOP_PATIENCE: 3 (was 5 - more aggressive)

**Features:** 22 minimal features (evidence-based from Jiang 2017, Zhang 2020)

**Data:** Weekly, per-ticker z-score normalized (52-week rolling window)

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import project modules
import config
from rl_system import create_walk_forward_folds, PortfolioEnv2

# Stable Baselines
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

print("✓ Imports complete")

✓ Loaded 26 features from metadata_weekly.json
  Using MINIMAL feature set (evidence-based)
✓ Imports complete


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 2. Verify Configuration

In [2]:
print("=" * 80)
print("TRIAL 5 CONFIGURATION")
print("=" * 80)

print(f"\nData:")
print(f"  Path: {config.DATA_PATH.name}")
print(f"  Features: {len(config.FEATURE_COLS)}")
print(f"  Using minimal features: {config.USE_MINIMAL_FEATURES}")

print(f"\nOutput:")
print(f"  Directory: {config.OUTPUT_DIR.name}")

print(f"\nTraining (Trial 5 - Improved Early Stopping):")
print(f"  LSTM hidden size: {config.LSTM_HIDDEN_SIZE} (Trial 2 best)")
print(f"  Eval frequency: {config.EVAL_FREQ:,} (was 2,500 - DOUBLED)")
print(f"  Early stop patience: {config.EARLY_STOP_PATIENCE} (was 5 - more aggressive)")
print(f"  Total timesteps: {config.TOTAL_TIMESTEPS:,}")
print(f"  Learning rate: {config.LSTM_LEARNING_RATE}")

print(f"\nExpected Behavior:")
max_evals = config.TOTAL_TIMESTEPS // config.EVAL_FREQ
print(f"  Max evaluations: {max_evals}")
print(f"  Stops after {config.EARLY_STOP_PATIENCE + 1} evals past peak (SB3 bug)")
print(f"  Steps past peak: {(config.EARLY_STOP_PATIENCE + 1) * config.EVAL_FREQ:,}")

print(f"\nEnvironment (weekly):")
print(f"  Rebalance frequency: {config.REBALANCE_FREQUENCY} weeks")
print(f"  Transaction cost: {config.TRANSACTION_COST*100:.2f}%")
print(f"  Reward type: {config.REWARD_TYPE}")

# Show first few features
print(f"\nFeature sample (first 10 of {len(config.FEATURE_COLS)}):")
for i, feat in enumerate(config.FEATURE_COLS[:10], 1):
    print(f"  {i:2d}. {feat}")

TRIAL 5 CONFIGURATION

Data:
  Path: train_weekly.parquet
  Features: 26
  Using minimal features: True

Output:
  Directory: models

Training (Trial 5 - Improved Early Stopping):
  LSTM hidden size: 50 (Trial 2 best)
  Eval frequency: 5,000 (was 2,500 - DOUBLED)
  Early stop patience: 3 (was 5 - more aggressive)
  Total timesteps: 100,000
  Learning rate: 0.0001

Expected Behavior:
  Max evaluations: 20
  Stops after 4 evals past peak (SB3 bug)
  Steps past peak: 20,000

Environment (weekly):
  Rebalance frequency: 4 weeks
  Transaction cost: 0.25%
  Reward type: log_return

Feature sample (first 10 of 26):
   1. momentum_1w_norm
   2. momentum_4w_norm
   3. momentum_13w_norm
   4. returns_52w_norm
   5. volatility_4w_norm
   6. volatility_52w_norm
   7. price_to_sma_12w_norm
   8. price_to_ema_26w_norm
   9. macd_histogram_norm
  10. bb_position_norm


## 3. Load Training Data

In [3]:
print("Loading training data...")
train_data = pd.read_parquet(config.DATA_PATH)

print(f"\nData shape: {train_data.shape}")
print(f"Date range: {train_data['date'].min()} to {train_data['date'].max()}")
print(f"Tickers: {sorted(train_data['ticker'].unique())}")
print(f"Weeks of data: {train_data['date'].nunique()}")

# Check for NaN in features
nan_counts = train_data[config.FEATURE_COLS].isna().sum()
if nan_counts.sum() > 0:
    print(f"\n⚠️ Warning: {nan_counts.sum()} total NaN values in features")
    print(f"Features with NaN:")
    for feat, count in nan_counts[nan_counts > 0].items():
        print(f"  - {feat}: {count}")
else:
    print("\n✓ No NaN values in features")

# Sample/feature ratio
n_samples = len(train_data)
n_features = len(config.FEATURE_COLS)
ratio = n_samples / n_features
print(f"\nSample/feature ratio: {ratio:.1f}")
if ratio >= 20:
    print("  ✓ EXCELLENT (>20 is ideal)")
elif ratio >= 10:
    print("  ✓ GOOD (10-20 is acceptable)")
else:
    print("  ⚠️ LOW (<10 has overfitting risk)")

Loading training data...

Data shape: (2632, 34)
Date range: 2016-06-23 00:00:00 to 2023-08-31 00:00:00
Tickers: ['AAPL', 'AMD', 'ASML', 'GOOG', 'MSFT', 'MU', 'NVDA']
Weeks of data: 376

✓ No NaN values in features

Sample/feature ratio: 101.2
  ✓ EXCELLENT (>20 is ideal)


## 4. Create Walk-Forward Folds

In [4]:
print("Creating walk-forward folds...")

# Create walk-forward folds with embargo gap
folds = create_walk_forward_folds(
    train_data=train_data,
    n_folds=config.N_FOLDS,
    min_train_size=None,  # Auto-calculate (max of 52 weeks or 20% of data)
    embargo_size=26,  # 6-month gap between train and val
    date_col='date'
)

print(f"\n✓ Created {len(folds)} folds")
for fold in folds:
    fold_idx = fold['fold_idx']
    train_fold = fold['train']
    val_fold = fold['val']
    train_dates = f"{train_fold['date'].min()} to {train_fold['date'].max()}"
    val_dates = f"{val_fold['date'].min()} to {val_fold['date'].max()}"
    print(f"\nFold {fold_idx}:")
    print(f"  Train: {len(train_fold):4d} samples | {train_dates}")
    print(f"  Val:   {len(val_fold):4d} samples | {val_dates}")
    print(f"  Embargo: {fold['embargo_periods']} weeks")

Creating walk-forward folds...

✓ Created 3 folds

Fold 0:
  Train:  525 samples | 2016-06-23 00:00:00 to 2017-11-23 00:00:00
  Val:    518 samples | 2018-05-31 00:00:00 to 2019-10-24 00:00:00
  Embargo: 26 weeks

Fold 1:
  Train: 1225 samples | 2016-06-23 00:00:00 to 2019-10-24 00:00:00
  Val:    518 samples | 2020-04-30 00:00:00 to 2021-09-23 00:00:00
  Embargo: 26 weeks

Fold 2:
  Train: 1925 samples | 2016-06-23 00:00:00 to 2021-09-23 00:00:00
  Val:    518 samples | 2022-03-31 00:00:00 to 2023-08-24 00:00:00
  Embargo: 26 weeks


## 5. Training Function

In [5]:
def train_lstm_ppo_fold(fold_idx, train_data, val_data):
    """
    Train LSTM PPO model on one fold.
    
    Trial 5 Configuration:
    - LSTM=50 (Trial 2 best config)
    - EVAL_FREQ=5000 (less frequent evaluation)
    - PATIENCE=3 (more aggressive stopping)
    
    Strategy: Give more time to learn between checks, stop more confidently
    """
    print(f"\n{'='*80}")
    print(f"TRAINING FOLD {fold_idx} (TRIAL 5)")
    print(f"{'='*80}")
    
    # Create output directory for this fold
    fold_dir = config.OUTPUT_DIR / f'fold_{fold_idx}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    
    # Create environments
    print("\nCreating environments...")
    train_env = PortfolioEnv2(
        data=train_data,
        feature_cols=config.FEATURE_COLS,
        tickers=config.TICKERS,
        rebalance_frequency=config.REBALANCE_FREQUENCY,
        transaction_cost=config.TRANSACTION_COST,
        max_weight_per_asset=config.MAX_WEIGHT_PER_ASSET,
        reward_lookback=config.REWARD_LOOKBACK,
        initial_capital=config.INITIAL_CAPITAL,
        reward_type=config.REWARD_TYPE,
        date_col='date',
        seed=config.RANDOM_SEED
    )
    
    val_env = PortfolioEnv2(
        data=val_data,
        feature_cols=config.FEATURE_COLS,
        tickers=config.TICKERS,
        rebalance_frequency=config.REBALANCE_FREQUENCY,
        transaction_cost=config.TRANSACTION_COST,
        max_weight_per_asset=config.MAX_WEIGHT_PER_ASSET,
        reward_lookback=config.REWARD_LOOKBACK,
        initial_capital=config.INITIAL_CAPITAL,
        reward_type=config.REWARD_TYPE,
        date_col='date',
        seed=config.RANDOM_SEED
    )
    
    print(f"✓ Train env: {len(train_data)} samples")
    print(f"✓ Val env: {len(val_data)} samples")
    
    # Setup callbacks
    print("\nSetting up callbacks...")
    stop_callback = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=config.EARLY_STOP_PATIENCE,
        min_evals=3,
        verbose=1
    )
    
    eval_callback = EvalCallback(
        val_env,
        best_model_save_path=str(fold_dir),
        log_path=str(fold_dir),
        eval_freq=config.EVAL_FREQ,
        n_eval_episodes=config.N_EVAL_EPISODES,
        deterministic=True,
        callback_after_eval=stop_callback,
        verbose=0
    )
    
    # Create LSTM PPO model
    print("\nCreating LSTM PPO model...")
    print(f"  LSTM hidden size: {config.LSTM_HIDDEN_SIZE} (Trial 2 best)")
    print(f"  Eval frequency: {config.EVAL_FREQ:,} (less frequent)")
    print(f"  Patience: {config.EARLY_STOP_PATIENCE} (stops at {config.EARLY_STOP_PATIENCE+1} due to SB3 bug)")
    print(f"  Learning rate: {config.LSTM_LEARNING_RATE}")
    print(f"  Min evals: 3 (warmup period)")
    
    model = RecurrentPPO(
        'MlpLstmPolicy',
        train_env,
        learning_rate=config.LSTM_LEARNING_RATE,
        n_steps=config.LSTM_N_STEPS,
        batch_size=config.LSTM_BATCH_SIZE,
        n_epochs=config.LSTM_N_EPOCHS,
        gamma=config.GAMMA,
        gae_lambda=config.GAE_LAMBDA,
        clip_range=config.CLIP_RANGE,
        ent_coef=config.LSTM_ENT_COEF,
        vf_coef=config.VF_COEF,
        max_grad_norm=config.MAX_GRAD_NORM,
        policy_kwargs=config.LSTM_POLICY_KWARGS,
        verbose=config.VERBOSE,
        tensorboard_log=str(fold_dir / 'tensorboard')
    )
    
    # Train
    print(f"\nTraining for {config.TOTAL_TIMESTEPS:,} timesteps...")
    model.learn(
        total_timesteps=config.TOTAL_TIMESTEPS,
        callback=eval_callback,
        progress_bar=True
    )
    
    print(f"\n✓ Fold {fold_idx} training complete")
    print(f"  Best model saved to: {fold_dir / 'best_model.zip'}")
    
    return model, fold_dir

print("✓ Training function defined")

✓ Training function defined


## 6. Train All Folds

In [6]:
# Train each fold
results = []

for fold in folds:
    fold_idx = fold['fold_idx']
    train_fold = fold['train']
    val_fold = fold['val']
    
    try:
        model, fold_dir = train_lstm_ppo_fold(fold_idx, train_fold, val_fold)
        results.append({
            'fold': fold_idx,
            'status': 'success',
            'fold_dir': fold_dir
        })
    except Exception as e:
        print(f"\n❌ Fold {fold_idx} failed: {e}")
        import traceback
        traceback.print_exc()
        results.append({
            'fold': fold_idx,
            'status': 'failed',
            'error': str(e)
        })

print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"Total folds: {len(results)}")
print(f"Successful: {sum(1 for r in results if r['status'] == 'success')}")
print(f"Failed: {sum(1 for r in results if r['status'] == 'failed')}")


TRAINING FOLD 0 (TRIAL 5)

Creating environments...
✓ Train env: 525 samples
✓ Val env: 518 samples

Setting up callbacks...

Creating LSTM PPO model...
  LSTM hidden size: 50 (Trial 2 best)
  Eval frequency: 5,000 (less frequent)
  Patience: 3 (stops at 4 due to SB3 bug)
  Learning rate: 0.0001
  Min evals: 3 (warmup period)


Output()


Training for 100,000 timesteps...


Stopping training because there was no new best model in the last 4 evaluations

Output()

...

Stopping training because there was no new best model in the last 4 evaluations

Output()


✓ Fold 1 training complete
  Best model saved to: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_1/best_model.zip

TRAINING FOLD 2 (TRIAL 5)

Creating environments...
✓ Train env: 1925 samples
✓ Val env: 518 samples

Setting up callbacks...

Creating LSTM PPO model...
  LSTM hidden size: 50 (Trial 2 best)
  Eval frequency: 5,000 (less frequent)
  Patience: 3 (stops at 4 due to SB3 bug)
  Learning rate: 0.0001
  Min evals: 3 (warmup period)

Training for 100,000 timesteps...


Stopping training because there was no new best model in the last 4 evaluations


✓ Fold 2 training complete
  Best model saved to: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_2/best_model.zip

TRAINING SUMMARY
Total folds: 3
Successful: 3
Failed: 0


## 7. Evaluation Results

In [7]:
# Load evaluation results from each fold
print("=" * 80)
print("TRIAL 5 EVALUATION RESULTS")
print("="  * 80)

for result in results:
    if result['status'] == 'success':
        fold_idx = result['fold']
        fold_dir = result['fold_dir']
        
        # Load evaluations.npz
        eval_file = fold_dir / 'evaluations.npz'
        if eval_file.exists():
            with np.load(eval_file) as data:
                timesteps = data['timesteps']
                results_array = data['results']
                ep_lengths = data['ep_lengths']
                
                # Calculate mean reward per evaluation
                mean_rewards = results_array.mean(axis=1)
                best_reward = mean_rewards.max()
                best_step = timesteps[mean_rewards.argmax()]
                final_reward = mean_rewards[-1]
                
                print(f"\nFold {fold_idx}:")
                print(f"  Best val reward: {best_reward:.4f} (at step {best_step:,})")
                print(f"  Final val reward: {final_reward:.4f}")
                print(f"  Evaluations: {len(timesteps)}")
                
                # Check for overfitting (significant drop from best to final)
                drop = (best_reward - final_reward) / abs(best_reward) * 100
                if drop > 40:
                    print(f"  ❌ Performance dropped {drop:.1f}% from peak (severe overfitting)")
                elif drop > 10:
                    print(f"  ⚠️ Performance dropped {drop:.1f}% from peak (moderate overfitting)")
                else:
                    print(f"  ✓ Stable performance (drop: {drop:.1f}%)")
        else:
            print(f"\nFold {fold_idx}: No evaluation file found")

print("\n" + "="*80)
print("Compare to Trial 2: Avg Sharpe 1.097, Val drops 20-66%")
print("="*80)

TRIAL 5 EVALUATION RESULTS

Fold 0:
  Best val reward: 0.3561 (at step 5,000)
  Final val reward: -0.2031
  Evaluations: 7
  ❌ Performance dropped 157.0% from peak (severe overfitting)

Fold 1:
  Best val reward: 0.8693 (at step 25,000)
  Final val reward: 0.7863
  Evaluations: 9
  ✓ Stable performance (drop: 9.5%)

Fold 2:
  Best val reward: 0.2268 (at step 5,000)
  Final val reward: 0.0812
  Evaluations: 7
  ❌ Performance dropped 64.2% from peak (severe overfitting)

Compare to Trial 2: Avg Sharpe 1.097, Val drops 20-66%


## 8. Save Summary

In [8]:
# Save training summary
summary = {
    'trial': 5,
    'description': 'LSTM=50, EVAL_FREQ=5000, PATIENCE=3',
    'feature_set': 'minimal_weekly',
    'n_features': len(config.FEATURE_COLS),
    'n_folds': len(results),
    'successful_folds': sum(1 for r in results if r['status'] == 'success'),
    'hyperparameters': {
        'lstm_hidden_size': config.LSTM_HIDDEN_SIZE,
        'eval_freq': config.EVAL_FREQ,
        'early_stop_patience': config.EARLY_STOP_PATIENCE,
        'learning_rate': config.LSTM_LEARNING_RATE,
        'total_timesteps': config.TOTAL_TIMESTEPS
    },
    'features': config.FEATURE_COLS
}

summary_file = config.OUTPUT_DIR / 'training_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Saved training summary to: {summary_file}")

print("\n" + "="*80)
print("TRAINING COMPLETE - TRIAL 5")
print("="*80)
print(f"\nNext steps:")
print(f"  1. Review evaluation results above")
print(f"  2. Compare to Trial 2 (Sharpe 1.097, Val drops 20-66%)")
print(f"  3. Run backtests if results look promising")
print(f"  4. Decision: Use Trial 5 if Sharpe ≥ 1.10, else revert to Trial 2")


✓ Saved training summary to: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/training_summary.json

TRAINING COMPLETE - TRIAL 5

Next steps:
  1. Review evaluation results above
  2. Compare to Trial 2 (Sharpe 1.097, Val drops 20-66%)
  3. Run backtests if results look promising
  4. Decision: Use Trial 5 if Sharpe ≥ 1.10, else revert to Trial 2
